# Checkpoint storage example for PyTorch DDP using Fashion MNIST Training

This example presents workflow of a checkpoint storage in the scratch-volume for a convolutional neural network (CNN) that classifies images using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)

The reference for the original notebook is [here](https://github.com/kubeflow/trainer/blob/master/examples/pytorch/image-classification/mnist.ipynb)

## Verification of dependencies and their versions

In [1]:
import kubeflow
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print("All imports successful!")

Kubeflow version: 0.3.0
All imports successful!


## Define the Training Function
Create function to train CNN model using Fashion MNIST data along with their config values (number of samples, epochs, etc)

In [2]:
def train_fashion_mnist(
    checkpoint_path="/scratch-volume/checkpoints_",
    save_interval=1,
    resume=False,
    epochs=50,
    batch_size=100,
    num_samples=1000,
    lr=0.1,
    momentum=0.9,
    seed=42,
):
    import os
    import random
    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler, Subset
    from torchvision import datasets, transforms

    # -----------------------------
    # Helpers
    # -----------------------------
    def is_distributed():
        return dist.is_available() and dist.is_initialized()

    def get_rank():
        return dist.get_rank() if is_distributed() else 0

    def is_main_process():
        return get_rank() == 0

    def barrier():
        if is_distributed():
            dist.barrier()

    def cleanup():
        if is_distributed():
            dist.destroy_process_group()

    def save_checkpoint(model, optimizer, epoch, path):
        os.makedirs(path, exist_ok=True)

        model_state = (
            model.module.state_dict()
            if hasattr(model, "module")
            else model.state_dict()
        )

        state = {
            "epoch": epoch,
            "model": model_state,
            "optimizer": optimizer.state_dict(),
        }

        latest_path = os.path.join(path, "latest.pt")
        epoch_path = os.path.join(path, f"epoch_{epoch:03d}.pt")

        tmp_latest = latest_path + ".tmp"
        tmp_epoch = epoch_path + ".tmp"

        torch.save(state, tmp_latest)
        torch.save(state, tmp_epoch)

        os.replace(tmp_latest, latest_path)
        os.replace(tmp_epoch, epoch_path)

        if is_main_process():
            print(f"Checkpoint saved at epoch {epoch} to {latest_path}")

    def load_checkpoint(model, optimizer, checkpoint_dir, device, local_rank):
        checkpoint_file = os.path.join(checkpoint_dir, "latest.pt")
        if not os.path.isfile(checkpoint_file):
            if is_main_process():
                print(f"No checkpoint found at {checkpoint_file}. Starting from scratch.")
            return 1

        if device.type == "cuda":
            map_location = {f"cuda:0": f"cuda:{local_rank}"}
        else:
            map_location = "cpu"

        checkpoint = torch.load(checkpoint_file, map_location=map_location)

        # Backward-compatible key handling
        model_key = "model" if "model" in checkpoint else "model_state"
        optim_key = "optimizer" if "optimizer" in checkpoint else "optimizer_state"

        model.load_state_dict(checkpoint[model_key])
        optimizer.load_state_dict(checkpoint[optim_key])

        start_epoch = checkpoint["epoch"] + 1

        if is_main_process():
            print(
                f"Resumed training from epoch {checkpoint['epoch']} "
                f"using {checkpoint_file}"
            )

        return start_epoch

    # -----------------------------
    # Reproducibility
    # -----------------------------
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # -----------------------------
    # Model definition
    # -----------------------------
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = torch.flatten(x, 1)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # -----------------------------
    # Device and distributed setup
    # -----------------------------
    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"
    local_rank = int(os.getenv("LOCAL_RANK", 0))

    distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if distributed:
        dist.init_process_group(backend=backend)

    if use_cuda:
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
    else:
        device = torch.device("cpu")

    if is_main_process():
        print(f"Using device: {device}, backend: {backend}")

    if distributed:
        print(
            f"WORLD_SIZE: {dist.get_world_size()}, "
            f"RANK: {dist.get_rank()}, LOCAL_RANK: {local_rank}"
        )
    else:
        if is_main_process():
            print("Running in single-process mode")

    # -----------------------------
    # Model and optimizer
    # -----------------------------
    model = Net().to(device)
    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank] if device.type == "cuda" else None,
            output_device=local_rank if device.type == "cuda" else None,
        )

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    # -----------------------------
    # Resume if requested
    # -----------------------------
    start_epoch = 1
    if resume:
        start_epoch = load_checkpoint(
            model=model,
            optimizer=optimizer,
            checkpoint_dir=checkpoint_path,
            device=device,
            local_rank=local_rank,
        )

    # -----------------------------
    # Dataset
    # -----------------------------
    transform = transforms.ToTensor()

    if is_main_process():
        datasets.FashionMNIST(
            root="./data",
            train=True,
            download=True,
            transform=transform,
        )

    barrier()

    dataset = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=False,
        transform=transform,
    )

    all_indices = list(range(len(dataset)))
    random.Random(seed).shuffle(all_indices)
    subset_indices = all_indices[:num_samples]
    subset_dataset = Subset(dataset, subset_indices)

    if distributed:
        sampler = DistributedSampler(
            subset_dataset,
            shuffle=True,
            drop_last=False,
        )
        shuffle = False
    else:
        sampler = None
        shuffle = True

    train_loader = DataLoader(
        subset_dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    barrier()

    # -----------------------------
    # Training loop
    # -----------------------------
    try:
        for epoch in range(start_epoch, epochs + 1):
            model.train()

            if distributed and sampler is not None:
                sampler.set_epoch(epoch)

            running_loss = 0.0

            for batch_idx, (inputs, labels) in enumerate(train_loader):
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                outputs = model(inputs)
                loss = F.nll_loss(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

                if batch_idx % 20 == 0 and is_main_process():
                    processed = batch_idx * len(inputs)
                    total = len(subset_dataset)
                    pct = 100.0 * batch_idx / max(1, len(train_loader))
                    print(
                        f"Train Epoch: {epoch} "
                        f"[{processed}/{total} ({pct:.0f}%)]\tLoss: {loss.item():.6f}"
                    )

            if is_main_process():
                avg_loss = running_loss / max(1, len(train_loader))
                print(f"Epoch {epoch} finished. Average loss: {avg_loss:.6f}")

            if is_main_process() and (epoch % save_interval == 0 or epoch == epochs):
                save_checkpoint(model, optimizer, epoch, checkpoint_path)

        barrier()

        if is_main_process():
            print("Training is finished")

    finally:
        cleanup()

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [3]:
from kubeflow.trainer import CustomTrainer, TrainerClient
client = TrainerClient()
trainer = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [4]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='retfound-image-generation', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/yukun-image-generation-runner:v0.0.8', num_nodes=1, device='gpu', device_count='1'), pretrained_model=None)
Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtim

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [5]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options

# ------------------------------------------------
# Configuration of resources
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "5",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

# Set github container registry
GITHUB_CONTAINER_REGISTRY = "ghcr.io/mxochicale/kubeflowtrainerimage/checkpointworflow:v0.0.2"

# Set up arguments of your function
FUNC_ARGS={
    "checkpoint_path": "/scratch-volume/YOUR_CHECKPOINT_PATH",
    "save_interval": 10,
    "resume": False,
    "epochs": 50,
    "batch_size": 128,
}


# ------------------------------------------------
# Seeting up to mount checkpoint volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/scratch-volume"

pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)
pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)

In [6]:
job_id = trainer.train(
    runtime=torch_runtime,
    trainer=kubeflow.trainer.CustomTrainer(
        func=train_fashion_mnist,
        func_args=FUNC_ARGS,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        image=GITHUB_CONTAINER_REGISTRY,
    ),
    options=[pod_template_overrides]
)

In [7]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: fe79783d9e75
Job Status: Created
Creation Time: 2026-03-30 11:14:23+00:00

Job details: TrainJob(name='fe79783d9e75', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 3, 30, 11, 14, 23, tzinfo=TzInfo(0)), status='Created')


In [8]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[11:14:23] Waiting... (1s)
[11:14:24] Waiting... (2s)
[11:14:26] Waiting... (3s)
[11:14:27] Waiting... (4s)
Logs received after 4 seconds:
  Using device: cuda:0, backend: nccl
  WORLD_SIZE: 1, RANK: 0, LOCAL_RANK: 0
100%|██████████| 26.4M/26.4M [00:01<00:00, 18.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 1.74MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 14.2MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 51.8MB/s]
  /opt/conda/lib/python3.11/site-packages/torch/distributed/distributed_c10d.py:4631: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
    warnings.warn(  # warn only once
  Train Epoch: 1 [0/1000 (0%)]	Loss: 2.303795
  Epoch 1 finished. Average loss: 2.288488
  Train Epoch: 2 [0/1000 (0%)]	Loss: 2.232576
  Epoch 2 finished. Average loss: 2.023522
  Train Epoch: 3 [0/1000 (0%)]	Loss: 2.791528
  Epoch 3 finished. Average loss: 1.841715
  Train Epoch: 4 [0/1000 (0%)]	Loss:

# Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [9]:
client.delete_job(job_id)